# Linear Model Selection and Regularization

Linear regression finds coefficients by minimizing error on the training data — but that's not always what we want. There are two main reasons to look for alternatives. First, **prediction accuracy**: if we have many predictors relative to observations, least squares overfits, and we can do better by shrinking or constraining the coefficients. Second, **interpretability**: some predictors are irrelevant, and least squares rarely sets any coefficient to exactly zero, so we end up with cluttered models that are hard to understand.

We'll explore three families of approaches: **Subset Selection** (choose the most relevant predictors and drop the rest), **Shrinkage / Regularization** (keep all predictors but penalize large coefficient values), and **Dimension Reduction** (combine predictors into a smaller set of new variables and regress on those).

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import itertools

import sklearn.linear_model as skl_lm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge, RidgeCV

import statsmodels.api as sm
import matplotlib.pyplot as plt

In [ ]:
url1 = 'https://raw.githubusercontent.com/dsahota-applied-data-analysis/data/main/Hitters.csv'
hitters = pd.read_csv(url1, index_col=0).dropna()

In [ ]:
hitters.head()

## Subset Selection Methods

Subset selection methods find the best combination of predictors to include in the model. The challenge is comparing models of different sizes — we can't just use training error, because adding more predictors always reduces it even if those predictors are useless. Instead, we either adjust the training error to penalize complexity (using metrics like AIC, BIC, or Adjusted R-squared), or we use cross-validation to directly estimate test error. When multiple model sizes perform similarly, the **one-standard-error rule** says to pick the simplest model whose error is within one standard error of the best model — favoring parsimony when the evidence doesn't strongly favor a more complex model.

In [ ]:
dummies = pd.get_dummies(hitters[['League', 'Division', 'NewLeague']])
dummies.info()
print(dummies.head())

In [ ]:
y = hitters.Salary
# Drop the column with the independent variable (Salary), and columns for which we created dummy variables
X_ = hitters.drop(['Salary', 'League', 'Division', 'NewLeague'], axis=1).astype('float64')
# Define the feature set X.
X = pd.concat([X_, dummies[['League_N', 'Division_W', 'NewLeague_N']]], axis=1)
X.head()

In [ ]:
X = X.astype('float64')

### Best Subset Selection

Best subset selection is exhaustive: try every possible combination of predictors, find the best model of each size, then compare across sizes to pick the winner. With p predictors, that means evaluating 2-to-the-p combinations — manageable for small p, but it gets very expensive fast. Here we cap the search at 7 predictors to keep it tractable.

In [ ]:
def processSubset(feature_set):
    # Fit model on feature_set and calculate RSS
    model = sm.OLS(y,X[list(feature_set)])
    regr = model.fit()
    RSS = ((regr.predict(X[list(feature_set)]) - y) ** 2).sum()
    return {"model":regr, "RSS":RSS}

In [ ]:
def getBest(k):
    results = []
    for combo in itertools.combinations(X.columns, k):
        results.append(processSubset(combo))
    # Wrap everything up in a nice dataframe
    models = pd.DataFrame(results)
    # Choose the model with the lowest RSS
    best_model = models.loc[models['RSS'].argmin()]
    print("Processed ", models.shape[0], "models on", k, "predictors.")
    # Return the best model
    return best_model

In [ ]:
models = pd.DataFrame(columns=["RSS", "model"])
for i in range(1,8):
    models.loc[i] = getBest(i)

In [ ]:
models

In [ ]:
models.loc[2,"model"].summary()

In [ ]:
models.loc[2,"model"].rsquared

In [ ]:
models.apply(lambda row: row.iloc[1].rsquared, axis=1)

In [ ]:
plt.figure(figsize=(20,10))
plt.rcParams.update({'font.size': 18, 'lines.markersize': 10})

# Set up a 2x2 grid so we can look at 4 plots at once
plt.subplot(2, 2, 1)

# We will now plot a red dot to indicate the model with the largest adjusted R^2 statistic.
# The argmax() function can be used to identify the location of the maximum point of a vector
plt.plot(models["RSS"])
plt.xlabel('# Predictors')
plt.ylabel('RSS')

rsquared = models.apply(lambda row: row[1].rsquared, axis=1)
plt.subplot(2, 2, 2)
plt.plot(rsquared)
plt.plot(rsquared.argmax() + 1, rsquared.max(), "or")
plt.xlabel('# Predictors')
plt.ylabel('adjusted rsquared')

# We'll do the same for AIC and BIC, this time looking for the models with the SMALLEST statistic
aic = models.apply(lambda row: row[1].aic, axis=1)

plt.subplot(2, 2, 3)
plt.plot(aic)
plt.plot(aic.argmin() + 1, aic.min(), "or")
plt.xlabel('# Predictors')
plt.ylabel('AIC')

bic = models.apply(lambda row: row[1].bic, axis=1)

plt.subplot(2, 2, 4)
plt.plot(bic)
plt.plot(bic.argmin() + 1, bic.min(), "or")
plt.xlabel('# Predictors')
plt.ylabel('BIC')

### Forward Stepwise Selection

Forward stepwise selection is a cheaper alternative to best subset. Start with no predictors. At each step, try adding each remaining predictor one at a time and keep whichever one improves the model the most. Repeat until you've built models of every size, then compare. This only evaluates around p-squared models instead of 2-to-the-p, which makes it practical with many predictors.

In [ ]:
def forward(predictors):
    # Pull out predictors we still need to process
    remaining_predictors = [p for p in X.columns if p not in predictors]

    results = []

    for p in remaining_predictors:
        results.append(processSubset(predictors+[p]))

    # Wrap everything up in a nice dataframe
    models = pd.DataFrame(results)

    # Choose the model with the lowest RSS
    best_model = models.loc[models['RSS'].argmin()]

    print("Processed ", models.shape[0], "models on", len(predictors)+1, "predictors.")

    # Return the best model
    return best_model

In [ ]:
models2 = pd.DataFrame(columns=["RSS", "model"])
predictors = []
for i in range(1,len(X.columns)+1):
    models2.loc[i] = forward(predictors)
    predictors = models2.loc[i]["model"].model.exog_names

In [ ]:
print(models2.loc[1, "model"].summary())
print(models2.loc[2, "model"].summary())

In [ ]:
print(models.loc[6, "model"].summary())
print(models2.loc[6, "model"].summary())

### Backward Stepwise Selection

Backward stepwise is the mirror image of forward: start with all predictors in the model and remove one at a time, always dropping the predictor whose removal hurts the model least. One constraint: you need more observations than predictors to fit the initial full model, so backward stepwise isn't usable in high-dimensional settings where forward stepwise still works.

In [ ]:
def backward(predictors):
    results = []
    for combo in itertools.combinations(predictors, len(predictors)-1):
        results.append(processSubset(combo))

    # Wrap everything up in a nice dataframe
    models = pd.DataFrame(results)

    # Choose the model with the lowest RSS
    best_model = models.loc[models['RSS'].argmin()]

    print("Processed ", models.shape[0], "models on", len(predictors)-1, "predictors.")

    # Return the best model
    return best_model

In [ ]:
models3 = pd.DataFrame(columns=["RSS", "model"], index = range(1,len(X.columns)))
predictors = X.columns
while(len(predictors) > 1):
    models3.loc[len(predictors)-1] = backward(predictors)
    predictors = models3.loc[len(predictors)-1]["model"].model.exog_names

In [ ]:
print(models.loc[7, "model"].params)
print(models2.loc[7, "model"].params)
print(models3.loc[7, "model"].params)

In [ ]:
models.loc[6,"model"].rsquared
models2.loc[6,"model"].rsquared
models3.loc[6,"model"].rsquared

## Shrinkage and Regularization

Shrinkage methods take a different approach: instead of removing predictors, we keep all of them but add a penalty for large coefficient values. This forces the model to keep coefficients small, which reduces overfitting. Before applying shrinkage, we **standardize the predictors** so each one has mean zero and standard deviation one — this ensures the penalty treats all predictors equally, regardless of their original units or scale.

### Ridge Regression

Ridge regression adds a penalty equal to the sum of squared coefficients, multiplied by a tuning parameter (called lambda, or alpha in sklearn). A larger tuning parameter means more shrinkage and a simpler model. Ridge never shrinks coefficients all the way to zero — it keeps every predictor in the model, just with smaller weights. We choose the tuning parameter using cross-validation.

Here, we will fit our Ridge Regression model with 100 different lambda values (called alpha in sklearn), then determine which corresponds to the most accurate model.

In [ ]:
# Standardize our Variables
scaler = StandardScaler().fit(X)

alphas = 10**np.linspace(10,-2,100)
ridge = skl_lm.Ridge()
coefs = []

for a in alphas:
    ridge.set_params(alpha=a)
    ridge.fit(scaler.transform(X), y)
    coefs.append(ridge.coef_)

In [ ]:
len(coefs)

In [ ]:
def Ridge_Regression(X, y, alpha):
    scaler = StandardScaler()
    ridge = skl_lm.Ridge()
    ridge.set_params(alpha=alpha)
    ridge.fit(scaler.fit_transform(X), y)
    return ridge.coef_

In [ ]:
Ridge_Regression(X, y, alpha=705)

#### Using RidgeCV to choose the optimal tuning parameter

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=22)

ridgecv = skl_lm.RidgeCV(alphas=alphas, scoring='neg_mean_squared_error')
ridgecv.fit(scaler.fit_transform(X_train), y_train)

In [ ]:
ridgecv.alpha_

The plot below shows how each coefficient changes as we increase regularization (move left to right = more shrinkage). All coefficients shrink toward zero, but at different rates. The dashed vertical line marks the alpha chosen by cross-validation. Notice that no coefficient ever reaches exactly zero — Ridge shrinks but never eliminates.

In [ ]:
coef_array = np.array(coefs)  # shape: (100 alphas, 19 features)

fig, ax = plt.subplots(figsize=(13, 6))
for j, col in enumerate(X.columns):
    ax.plot(np.log10(alphas), coef_array[:, j], label=col, linewidth=1.5)
ax.axvline(np.log10(ridgecv.alpha_), color='black', linestyle='--', linewidth=1.5,
           label=f'CV-chosen α = {ridgecv.alpha_:.1f}')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xlabel('log₁₀(α)  —  ← less regularization  |  more regularization →', fontsize=11)
ax.set_ylabel('Coefficient value')
ax.set_title('Ridge Regression: coefficient paths as regularization increases')
ax.legend(loc='upper right', fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

Now that we have our optimal tuning parameter (``ridgecv.alpha_=76``), we can plug that into our Ridge Regression in order to minimize the Test MSE.

In [ ]:
optimal_ridge = skl_lm.Ridge()
optimal_ridge.set_params(alpha=ridgecv.alpha_)
optimal_ridge.fit(scaler.fit_transform(X_train), y_train)

In [ ]:
mean_squared_error(y_test, optimal_ridge.predict(scaler.transform(X_test)))

In [ ]:
pd.Series(optimal_ridge.coef_.flatten(), index=X.columns)

### The Lasso

Lasso is similar to Ridge but uses the sum of the absolute values of the coefficients as the penalty instead of the sum of squares. That one difference has a big consequence: Lasso can shrink coefficients all the way to exactly zero, effectively removing predictors from the model. This makes Lasso useful for feature selection — with a large enough tuning parameter, many coefficients will drop to zero and you get a sparse model that's easy to interpret.

First, let's find the optimal alpha using cross validation.

In [ ]:
lassocv = skl_lm.LassoCV(alphas = alphas, max_iter=10000)
lassocv.fit(scaler.fit_transform(X_train), y_train.values.ravel())

In [ ]:
lassocv.alpha_

In [ ]:
lasso = skl_lm.Lasso()
lasso.set_params(alpha = lassocv.alpha_)
print('\n')
lasso.fit(scaler.fit_transform(X_train), y_train.values.ravel())
print('\n')
mean_squared_error(y_test, lasso.predict(scaler.transform(X_test)))

As we can see by the Test MSE, Lasso performs slightly better than Ridge Regression on this dataset.

In [ ]:
pd.Series(lasso.coef_.flatten(), index = X.columns)

The plot below shows how Lasso coefficients change as regularization increases. Unlike Ridge, Lasso drives coefficients to **exactly zero** — notice the sharp kinks where a predictor's line hits zero and stays there. This is what makes Lasso perform automatic feature selection.

In [ ]:
lasso_alphas = np.linspace(lassocv.alpha_ * 0.01, lassocv.alpha_ * 3, 100)
lasso_coef_paths = []
_scaler = StandardScaler().fit(X_train)

for a in lasso_alphas:
    _l = skl_lm.Lasso(alpha=a, max_iter=10000)
    _l.fit(_scaler.transform(X_train), y_train.values.ravel())
    lasso_coef_paths.append(_l.coef_.copy())

lasso_coef_array = np.array(lasso_coef_paths)

fig, ax = plt.subplots(figsize=(13, 6))
for j, col in enumerate(X.columns):
    ax.plot(lasso_alphas, lasso_coef_array[:, j], label=col, linewidth=1.5)
ax.axvline(lassocv.alpha_, color='black', linestyle='--', linewidth=1.5,
           label=f'CV-chosen α = {lassocv.alpha_:.4f}')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xlabel('α  —  ← less regularization  |  more regularization →', fontsize=11)
ax.set_ylabel('Coefficient value')
ax.set_title('Lasso: coefficient paths — predictors drop to zero as regularization increases')
ax.legend(loc='upper right', fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

Interestingly, Lasso Regression cancels out many of the predictors, using only 6 of the original 19.

## Dimension Reduction: PCR and PLS Regression

Dimension reduction methods take a different approach entirely. Instead of selecting or shrinking the original predictors, we transform them into a smaller set of new variables — called components or directions — and then fit a regression on those. This reduces the number of things we need to estimate, which helps when we have many predictors relative to the amount of data.

A note on high-dimensional settings: when you have more predictors than observations, least squares completely breaks down — it can perfectly fit the training data by passing through every point, but generalizes terribly. In those situations, regularization or dimension reduction becomes essential rather than optional.

### Principal Components Regression (PCR)

PCR finds the directions in the predictor data along which the observations vary the most — called principal components — and then regresses the response on those. The components are combinations of all the original predictors. One key feature: PCR is **unsupervised**, meaning the response variable is not used to find the components. We use cross-validation to choose how many components to keep.

In [ ]:
#PCR to predict salary
pca = PCA()
X_train_reduced = pca.fit_transform(scaler.fit_transform(X_train))

# We will use OLS to fit our M-dimensional data, derived from PCA
regr = skl_lm.LinearRegression()

# 10-fold CV, with shuffle
kf_10 = KFold(n_splits=10, shuffle=True, random_state=1)
mse = []

for i in np.arange(1, 20):
    score = -1*cross_val_score(regr, X_train_reduced[:,:i], y_train, cv=kf_10, scoring='neg_mean_squared_error').mean()
    mse.append(score)

mse_per_component = pd.Series(np.array(mse).flatten(), index = np.arange(1,20))
print(mse_per_component)

The two plots below help us choose how many principal components to keep. The left panel shows the fraction of total variance in the predictors explained by each additional component (a scree plot). The right panel shows cross-validated MSE as a function of component count — we want the elbow where adding more components stops helping.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: explained variance ratio (scree plot)
explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)
axes[0].bar(range(1, len(explained)+1), explained, color='steelblue', alpha=0.7, label='Individual')
axes[0].plot(range(1, len(explained)+1), cumulative, 'o-', color='darkorange', label='Cumulative')
axes[0].axhline(0.9, color='gray', linestyle='--', linewidth=1, label='90% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot: variance explained per component')
axes[0].legend()
axes[0].set_xticks(range(1, len(explained)+1))

# Right: CV MSE vs number of components
best_m = mse_per_component.idxmin()
axes[1].plot(mse_per_component.index, mse_per_component.values, 'o-', color='steelblue')
axes[1].axvline(best_m, color='crimson', linestyle='--', linewidth=1.5,
                label=f'Best M = {best_m}  (MSE = {mse_per_component[best_m]:.0f})')
axes[1].set_xlabel('Number of Principal Components (M)')
axes[1].set_ylabel('Cross-Validated MSE')
axes[1].set_title('PCR: CV MSE vs number of components')
axes[1].legend()
axes[1].set_xticks(mse_per_component.index)

plt.tight_layout()
plt.show()

In [ ]:
np.amin(mse_per_component)
# This lets us know that the model with 5 principal components has the smallest training error

##### Transform test data with PCA loadings and fit regression on 5 principal components

In [ ]:
# optimal_pca = PCA(n_components=5)
X_test_reduced = pca.transform(scaler.transform(X_test))[:, :5]

# train OLS model on PCA-reduced training data
pca_regr = skl_lm.LinearRegression()
pca_regr.fit(X_train_reduced[:,:5], y_train.values.ravel())
print('\n')

# Make predictions with our model and calculate MSE
y_pred = pca_regr.predict(X_test_reduced)
mean_squared_error(y_test.values.ravel(), y_pred)

### Partial Least Squares (PLS)

Partial Least Squares is like PCR but **supervised**: it finds directions that explain both the variation in the predictors and the relationship to the response variable. By incorporating information about what we're trying to predict, PLS can sometimes find more relevant components than PCR. In practice, the two methods often perform similarly on real datasets.

In [ ]:
pls = PLSRegression()
pls.fit(scaler.fit_transform(X_train), y_train)
print('\n')

mean_squared_error(y_test, pls.predict(scaler.transform(X_test)))

### Comparing All Five Methods

Let's put all five approaches side by side — ordinary least squares as the baseline, plus Ridge, Lasso, PCR, and PLS. Test MSE is our measure, since we held out the same 50% test set throughout.

In [ ]:
# Recompute OLS test MSE on the same train/test split
ols_model = skl_lm.LinearRegression()
ols_model.fit(scaler.fit_transform(X_train), y_train.values.ravel())
ols_mse = mean_squared_error(y_test, ols_model.predict(scaler.transform(X_test)))

ridge_mse  = mean_squared_error(y_test, optimal_ridge.predict(scaler.transform(X_test)))
lasso_mse  = mean_squared_error(y_test, lasso.predict(scaler.transform(X_test)))
pcr_mse    = mean_squared_error(y_test.values.ravel(), y_pred)
pls_mse    = mean_squared_error(y_test, pls.predict(scaler.transform(X_test)))

methods    = ['OLS', 'Ridge', 'Lasso', 'PCR', 'PLS']
mse_values = [ols_mse, ridge_mse, lasso_mse, pcr_mse, pls_mse]

colors = ['#7f7f7f', '#4a90d9', '#50a86f', '#e8963a', '#b55bb5']
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(methods, mse_values, color=colors, alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, mse_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1000,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Test MSE')
ax.set_title('Test MSE comparison across all five model selection methods\n(Hitters dataset, 50% test split)')
ax.set_ylim(0, max(mse_values) * 1.15)
plt.tight_layout()
plt.show()

print(f'\nOLS:   {ols_mse:,.0f}')
print(f'Ridge: {ridge_mse:,.0f}')
print(f'Lasso: {lasso_mse:,.0f}')
print(f'PCR:   {pcr_mse:,.0f}')
print(f'PLS:   {pls_mse:,.0f}')